# Publication notes

This is the original exploratory workflow with saved outputs removed and paths made configurable. Collection and analytical logic is retained. Source websites and identifiers may change; notebooks 01–02 are an inspected research workflow, not a guaranteed unattended scraper. Review collection permissions and source-specific settings before recollection. Do not run against your only copy of historical data: original export cells can overwrite files.

Run from a clean kernel in order. Data is local and excluded from GitHub. Historical saved review text and reviewer details are not published as notebook outputs.

The final two cells retain the earlier, multi-category `product_claims.csv` workflow for historical context. They are superseded for the current dashboard by notebook 06 and its one-category-per-claim tables. Run only the first four original code cells for current review-theme exports.


In [ ]:
from pathlib import Path
import os

# Configure KBEAUTY_PROJECT_ROOT to keep data outside the cloned repository.
_working_dir = Path.cwd().resolve()
_default_root = _working_dir.parent if _working_dir.name == "notebooks" else _working_dir
PROJECT_ROOT = Path(os.environ.get("KBEAUTY_PROJECT_ROOT", str(_default_root))).expanduser().resolve()
# Original collection cells use relative data/ paths.
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd
import re
import html

project_folder = PROJECT_ROOT
cleaned_folder = project_folder / "data" / "cleaned" / "all_products"

tables = {}

for name in [
    "products",
    "reviews",
    "ingredients",
    "review_skin_concerns"
]:
    tables[name] = pd.read_csv(
        cleaned_folder / f"{name}.csv",
        dtype={
            "product_id": "string",
            "review_id": "string",
            "sku": "string"
        }
    )

products = tables["products"]
reviews = tables["reviews"]
ingredients = tables["ingredients"]
concerns = tables["review_skin_concerns"]

# Confirm keys and relationships.
assert len(products) == 12
assert len(reviews) == 3600
assert products["product_id"].is_unique
assert reviews["review_id"].is_unique
assert reviews["review_id"].notna().all()

assert reviews["product_id"].isin(products["product_id"]).all()
assert ingredients["product_id"].isin(products["product_id"]).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()

# Prepare text without changing original titles or review bodies.
reviews["theme_analysis_text"] = (
    reviews["review_title"].fillna("")
    + ". "
    + reviews["review_text"].fillna("")
).map(html.unescape).str.replace(r"\s+", " ", regex=True).str.strip()

language = (
    reviews["language"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

english_mask = language.eq("en").fillna(False)
has_text = reviews["review_text"].fillna("").str.strip().ne("")

reviews["theme_analysis_status"] = "Not assessed: language"
reviews.loc[english_mask & ~has_text, "theme_analysis_status"] = (
    "Not assessed: missing review text"
)
reviews.loc[english_mask & has_text, "theme_analysis_status"] = (
    "Ready for tagging"
)

# Record an explicit eligibility flag for Tableau denominators.
reviews["theme_analysis_eligible"] = english_mask & has_text

print("TABLE COUNTS")
for name, table in tables.items():
    print(f"{name}: {len(table):,}")

print("\nTHEME ANALYSIS STATUS")
print(reviews["theme_analysis_status"].value_counts().to_string())

print("\nAll starting checks passed.")

In [ ]:
# These rules identify mentions, not sentiment or product causation.
theme_patterns = {
    "Hydration": (
        r"\b(?:hydrat\w*|dehydrat\w*|moistur\w*|"
        r"dry|dryness|drier|driest|drying|flaky|flakiness)\b"
    ),
    "Texture": (
        r"\b(?:textur\w*|sticky|stickiness|tacky|tackiness|"
        r"greasy|greasiness|oily|oiliness|lightweight|"
        r"light[\s-]+weight|heavy|heaviness|thick|thickness|"
        r"watery|silky|silkiness|absorb\w*|absorption|"
        r"residue|pilling|pills|consistency)\b"
    ),
    "Irritation / breakouts": (
        r"\b(?:irritat\w*|sting\w*|burn\w*|redness|"
        r"itch\w*|rash\w*|breakouts?|break[\s-]*out|"
        r"broke\s+(?:me\s+)?out|acne|pimples?|"
        r"blemish\w*|eczema|rosacea|sensitiv\w*)\b"
    ),
    "Packaging": (
        r"\b(?:packag\w*|bottles?|pumps?|dispens\w*|"
        r"containers?|caps?|lids?|jars?|tubes?|"
        r"leak\w*|spill\w*|spray\w*|mist\w*|nozzles?)\b"
    ),
    "Price / value": (
        r"\b(?:price\w*|pricy|pricing|expensive|inexpensive|"
        r"afford\w*|cheap\w*|cost|costs|costly|"
        r"overpriced|value|money|budget\w*|"
        r"worth\s+(?:it|every|the|my|your))\b"
    )
}

compiled_patterns = {
    theme: re.compile(pattern, flags=re.IGNORECASE)
    for theme, pattern in theme_patterns.items()
}

theme_rows = []

eligible_reviews = reviews.loc[
    reviews["theme_analysis_eligible"],
    ["review_id", "theme_analysis_text"]
]

for row in eligible_reviews.itertuples(index=False):
    text = row.theme_analysis_text

    for theme, pattern in compiled_patterns.items():
        matches = list(pattern.finditer(text))
        if not matches:
            continue

        # Keep context around the first match for quick inspection.
        first_match = matches[0]
        start = max(0, first_match.start() - 100)
        end = min(len(text), first_match.end() + 180)

        theme_rows.append({
            "review_id": row.review_id,
            "theme": theme,
            "matched_terms": "; ".join(sorted({
                match.group(0).casefold() for match in matches
            })),
            "evidence_excerpt": text[start:end],
            "tagging_method": "Keyword mention rules v1",
            "validation_status": "Not manually reviewed"
        })

review_themes = pd.DataFrame(
    theme_rows,
    columns=[
        "review_id", "theme", "matched_terms",
        "evidence_excerpt", "tagging_method", "validation_status"
    ]
)

assert not review_themes.duplicated(["review_id", "theme"]).any()
assert review_themes["review_id"].isin(eligible_reviews["review_id"]).all()

tagged_ids = set(review_themes["review_id"])

reviews.loc[
    reviews["theme_analysis_eligible"],
    "theme_analysis_status"
] = "Assessed: no theme matched"

reviews.loc[
    reviews["theme_analysis_eligible"]
    & reviews["review_id"].isin(tagged_ids),
    "theme_analysis_status"
] = "Assessed: theme matched"

# Create a small, reproducible validation sample:
# two matched reviews per brand per theme, where available.
theme_candidates = review_themes.merge(
    reviews[["review_id", "product_id", "rating"]],
    on="review_id",
    validate="many_to_one"
).merge(
    products[["product_id", "brand"]],
    on="product_id",
    validate="many_to_one"
)

sample_parts = [
    group.sample(n=min(2, len(group)), random_state=42)
    for _, group in theme_candidates.groupby(["theme", "brand"])
]

theme_validation_sample = (
    pd.concat(sample_parts, ignore_index=True)
    if sample_parts else theme_candidates.copy()
)

print("CANDIDATE THEME COUNTS — NOT YET VALIDATED")
print(review_themes["theme"].value_counts().to_string())

print("\nREVIEW COVERAGE")
print(reviews["theme_analysis_status"].value_counts().to_string())

print("\nVALIDATION SAMPLE")
for row in theme_validation_sample.itertuples(index=False):
    print(f"\n[{row.theme}] {row.brand} | {row.review_id}")
    print("Matched terms:", row.matched_terms)
    print("Excerpt:", row.evidence_excerpt)

In [ ]:
theme_patterns = {
    "Hydration": (
        r"\b(?:hydrat\w*|dehydrat\w*|moisturi[sz]\w*|moisture|"
        r"dry|dryness|drier|driest|drying|flaky|flakiness)\b"
    ),
    "Texture / feel": (
        r"\b(?:textur\w*|sticky|stickiness|tacky|tackiness|"
        r"greasy|greasiness|lightweight|light[\s-]+weight\w*|"
        r"heavy|heaviness|thick|thickness|watery|silky|"
        r"absorb\w*|absorption|residue|pilling|consistency)\b"
        r"|\bsoaks?\s+in\b"
    ),
    "Sensitivity / breakouts": (
        r"\b(?:irritat\w*|sting\w*|burn\w*|redness|"
        r"itch\w*|rash\w*|breakouts?|break[\s-]*out|"
        r"broke\s+(?:me\s+)?out|acne|pimples?|"
        r"blemish\w*|eczema|rosacea|sensitiv\w*|dermatitis)\b"
    ),
    "Packaging / format": (
        r"\b(?:packag\w*|bottles?|pumps?|dispens\w*|"
        r"containers?|caps?|lids?|jars?|tubes?|"
        r"leak\w*|spill\w*|spray\w*|mist\w*|nozzles?)\b"
    ),
    "Price / value": (
        r"\b(?:prices?|priced|pricey|pricy|pricing|expensive|"
        r"inexpensive|afford\w*|cheap\w*|cost|costs|costly|"
        r"overpriced|value|money|budget\w*|refund\w*|"
        r"discount\w*|bargain\w*)\b"
        r"|\bworth\s+(?:it|every\s+penny|the\s+(?:price|cost|money))\b"
        r"|\bon\s+sale\b"
    )
}

compiled_patterns = {
    theme: re.compile(pattern, re.IGNORECASE)
    for theme, pattern in theme_patterns.items()
}

theme_rows = []

for row in eligible_reviews.itertuples(index=False):
    text = row.theme_analysis_text

    for theme, pattern in compiled_patterns.items():
        matches = list(pattern.finditer(text))
        if not matches:
            continue

        first = matches[0]
        theme_rows.append({
            "review_id": row.review_id,
            "theme": theme,
            "matched_terms": "; ".join(sorted({
                match.group(0).casefold() for match in matches
            })),
            "evidence_excerpt": text[
                max(0, first.start() - 100):first.end() + 180
            ],
            "tagging_method": "Keyword mention rules v2",
            "validation_status": "Not manually reviewed"
        })

review_themes = pd.DataFrame(theme_rows)

# One observed false positive: the packaging mention refers to La Mer.
# Preserve the exclusion as an explicit manual decision.
theme_exclusions = pd.DataFrame([{
    "review_id": "860333b6-0602-4978-af77-a522ac099d8a",
    "theme": "Packaging / format",
    "exclusion_reason": "Sampled packaging reference concerns another brand"
}])

review_themes = review_themes.merge(
    theme_exclusions[["review_id", "theme"]].assign(exclude=True),
    on=["review_id", "theme"],
    how="left",
    validate="one_to_one"
)

review_themes = (
    review_themes[review_themes["exclude"].isna()]
    .drop(columns="exclude")
    .reset_index(drop=True)
)

assert not review_themes.duplicated(["review_id", "theme"]).any()
assert review_themes["review_id"].isin(eligible_reviews["review_id"]).all()

tagged_ids = set(review_themes["review_id"])

reviews.loc[
    reviews["theme_analysis_eligible"], "theme_analysis_status"
] = "Assessed: no theme matched"

reviews.loc[
    reviews["theme_analysis_eligible"]
    & reviews["review_id"].isin(tagged_ids),
    "theme_analysis_status"
] = "Assessed: theme matched"

print("REVISED CANDIDATE COUNTS")
print(review_themes["theme"].value_counts().to_string())

# Inspect three untagged English reviews per brand.
untagged = reviews[
    reviews["theme_analysis_eligible"]
    & ~reviews["review_id"].isin(tagged_ids)
].merge(
    products[["product_id", "brand"]],
    on="product_id",
    validate="many_to_one"
)

print("\nUNTAGGED REVIEW SAMPLE")
for brand, group in untagged.groupby("brand"):
    for row in group.sample(
        n=min(3, len(group)), random_state=42
    ).itertuples(index=False):
        print(f"\n{brand} | {row.review_id}")
        print(row.theme_analysis_text)

In [ ]:
import json
from pathlib import Path

# 1. Add the phrases identified in the omission check.
theme_patterns["Texture / feel"] += (
    r"|\bvery\s+light\b"
    r"|\blight\s+(?:feeling|feel|texture|consistency)\b"
    r"|\bfeels?\s+light\b"
)
theme_patterns["Sensitivity / breakouts"] += r"|\bgentle\b|\bgentleness\b"

compiled_patterns = {
    theme: re.compile(pattern, re.IGNORECASE)
    for theme, pattern in theme_patterns.items()
}

excluded_pairs = set(
    theme_exclusions[["review_id", "theme"]]
    .itertuples(index=False, name=None)
)

# 2. Rebuild the final review–theme table.
theme_rows = []

for row in eligible_reviews.itertuples(index=False):
    text = row.theme_analysis_text

    for theme, pattern in compiled_patterns.items():
        if (row.review_id, theme) in excluded_pairs:
            continue

        matches = list(pattern.finditer(text))
        if not matches:
            continue

        first = matches[0]
        theme_rows.append({
            "review_id": row.review_id,
            "theme": theme,
            "matched_terms": "; ".join(sorted({
                match.group(0).casefold() for match in matches
            })),
            "evidence_excerpt": text[
                max(0, first.start() - 100):first.end() + 180
            ],
            "tagging_method": "Keyword mention rules v3",
            "validation_status": "Exploratory; not individually validated"
        })

review_themes = pd.DataFrame(theme_rows)

tagged_ids = set(review_themes["review_id"])
reviews.loc[
    reviews["theme_analysis_eligible"], "theme_analysis_status"
] = "Assessed: no theme matched"
reviews.loc[
    reviews["theme_analysis_eligible"]
    & reviews["review_id"].isin(tagged_ids),
    "theme_analysis_status"
] = "Assessed: theme matched"

# 3. Prepare Tableau-friendly dates without assigning unknown timezones.
# Retain review_date as the original source timestamp.
reviews["review_calendar_date"] = reviews["review_date"].map(
    lambda value: pd.Timestamp(value).date().isoformat()
)

# Restore nullable booleans explicitly after CSV loading.
boolean_columns = [
    "verified_standardized",
    "incentivized_standardized",
    "sponsored_standardized",
    "is_english"
]

for column in boolean_columns:
    if column in reviews.columns:
        keys = reviews[column].astype("string").str.strip().str.casefold()
        unexpected = set(keys.dropna()) - {"true", "false"}
        assert not unexpected, f"Unexpected values in {column}: {unexpected}"
        reviews[column] = keys.map({
            "true": True, "false": False
        }).astype("boolean")

# 4. Validate all relationships.
assert len(products) == 12
assert len(reviews) == 3600
assert products["product_id"].is_unique
assert reviews["review_id"].notna().all()
assert reviews["review_id"].is_unique
assert reviews["product_id"].isin(products["product_id"]).all()
assert ingredients["product_id"].isin(products["product_id"]).all()
assert concerns["review_id"].isin(reviews["review_id"]).all()
assert review_themes["review_id"].isin(eligible_reviews["review_id"]).all()
assert not review_themes.duplicated(["review_id", "theme"]).any()
assert not ingredients.duplicated(
    ["product_id", "ingredient_position"]
).any()
assert not concerns.duplicated(
    ["review_id", "skin_concern_standardized"]
).any()

# 5. Export the five relational tables.
tableau_folder = project_folder / "data" / "tableau"
tableau_folder.mkdir(parents=True, exist_ok=True)

tableau_tables = {
    "products": products,
    "reviews": reviews.drop(columns=["theme_analysis_text"]),
    "ingredients": ingredients,
    "review_skin_concerns": concerns,
    "review_themes": review_themes
}

for name, table in tableau_tables.items():
    file_path = tableau_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    saved = pd.read_csv(
        file_path,
        dtype={"product_id": "string", "review_id": "string", "sku": "string"}
    )
    assert len(saved) == len(table)
    assert saved.columns.tolist() == table.columns.tolist()
    print(f"{name}.csv: {len(saved):,} rows saved")

# 6. Save reproducibility notes separately from the Tableau tables.
method_folder = project_folder / "data" / "analysis" / "theme_method"
method_folder.mkdir(parents=True, exist_ok=True)

with open(method_folder / "theme_rules_v3.json", "w") as file:
    json.dump(theme_patterns, file, indent=2)

theme_exclusions.to_csv(
    method_folder / "manual_exclusions.csv", index=False
)
theme_validation_sample.to_csv(
    method_folder / "initial_matched_sample.csv", index=False
)

method_notes = """Theme analysis: exploratory keyword mentions, version 3.
Scope: reviews marked English with nonblank review text.
Non-English and unknown-language reviews remain in Reviews but are not assessed.
One row per review–theme pair. Multiple themes per review are allowed.
Mentions do not establish sentiment, complaints, effectiveness, or causation.
Sensitivity includes gentle/sensitive-skin discussion and negated reactions.
Texture includes product feel and skin texture.
Packaging includes neutral references to formats and containers.
Initial QA inspected 30 matched excerpts and 9 unmatched reviews.
Rules were revised after this small sample; accuracy/recall were not estimated.
No complete manual validation or independent holdout evaluation was performed.
Reviews without matches are not proof that none of the themes is present.
Product prices and formulas describe selected variants at collection;
product-level reviews can cover other variants and historical formulations.
"""

(method_folder / "README.txt").write_text(method_notes)

print("\nFINAL THEME COUNTS")
print(review_themes["theme"].value_counts().to_string())

print("\nANALYSIS COVERAGE")
print(reviews["theme_analysis_status"].value_counts().to_string())

print("\nAll export checks passed.")
print("Tableau folder:", tableau_folder.resolve())

In [ ]:
from pathlib import Path
import pandas as pd

products_for_claims = pd.read_csv(
    (PROJECT_ROOT / "data/tableau/products.csv"),
    dtype={"product_id": "string"}
)

for _, row in products_for_claims.iterrows():
    print(f"\n{row['product_id']} | {row['product_name']}")
    print(row["key_benefits"])

In [ ]:
# Manually categorize the collected brand claims.
# These categories describe brand promises, not verified effects.

claim_mapping = {
    "BOJ_CLEANSER_01": [
        ("Slightly Acidic", "pH positioning"),
        ("Hydrating Cleansing", "Hydration"),
        ("Hydrating Cleansing", "Cleansing"),
    ],
    "BOJ_MOISTURIZER_01": [
        ("Long Lasting Moisture", "Hydration"),
        ("Glass Skin Glow", "Radiance / glow"),
    ],
    "BOJ_SERUM_01": [
        ("Manages Sebum & Moisture", "Oil control"),
        ("Manages Sebum & Moisture", "Hydration"),
    ],
    "BOJ_TONER_01": [
        ("Long Lasting Hydration", "Hydration"),
        ("Skin Calming Effect", "Soothing"),
    ],
    "COSRX_CLEANSER_01": [
        ("Hydrating cleanse without stripping", "Hydration"),
        ("Hydrating cleanse without stripping", "Cleansing"),
        ("Removes impurities", "Cleansing"),
        ("Protects skin barrier", "Barrier support"),
        ("Restores pH Balance", "pH positioning"),
    ],
    "COSRX_MOISTURIZER_01": [
        ("Boosts skin radiance and visibly enhances skin clarity",
         "Radiance / glow"),
        ("Soothes the appearance of irritated skin", "Soothing"),
        ("Provides long-lasting and deep hydration", "Hydration"),
    ],
    "COSRX_SERUM_01": [
        ("Firmness", "Firmness / elasticity"),
        ("Radiance", "Radiance / glow"),
        ("Smoothness", "Skin texture / smoothness"),
        ("Deep hydration", "Hydration"),
        ("Glow", "Radiance / glow"),
    ],
    "COSRX_TONER_01": [
        ("Nourish for healthy skin", "Nourishment"),
        ("Plump, glowing skin", "Plumping"),
        ("Plump, glowing skin", "Radiance / glow"),
        ("Reduce redness", "Redness reduction"),
        ("Improve skin texture", "Skin texture / smoothness"),
    ],
    "LANEIGE_CLEANSER_01": [
        ("Removes impurities, face makeup and sunscreen", "Cleansing"),
        ("Hydrates", "Hydration"),
        ("Soothes", "Soothing"),
        ("Supports the moisture barrier", "Barrier support"),
    ],
    "LANEIGE_MOISTURIZER_01": [
        ("Barrier support", "Barrier support"),
        ("Intense, long-lasting hydration", "Hydration"),
    ],
    "LANEIGE_SERUM_01": [
        ("Intense hydration", "Hydration"),
        ("Smooth, supple complexion", "Skin texture / smoothness"),
        ("Visibly firm and improve elasticity", "Firmness / elasticity"),
    ],
    "LANEIGE_TONER_01": [
        ("Hydration", "Hydration"),
        ("Barrier support", "Barrier support"),
        ("Nourish and visibly firm", "Nourishment"),
        ("Nourish and visibly firm", "Firmness / elasticity"),
        ("Soothe", "Soothing"),
    ],
}

assert products_for_claims["product_id"].is_unique
assert set(claim_mapping) == set(products_for_claims["product_id"])

rows = [
    {
        "product_id": product_id,
        "claim_text": claim_text,
        "claim_category": category,
    }
    for product_id, claims in claim_mapping.items()
    for claim_text, category in claims
]

product_claims = pd.DataFrame(rows)

# Retain the complete collected benefits text and product-page URL.
source = products_for_claims[
    ["product_id", "key_benefits", "product_url"]
].rename(columns={
    "key_benefits": "original_key_benefits",
    "product_url": "source_product_url",
})

product_claims = product_claims.merge(
    source,
    on="product_id",
    how="left",
    validate="many_to_one"
)

product_claims["coding_method"] = "Manual categorization of collected key benefits"

assert product_claims[
    ["product_id", "claim_text", "claim_category"]
].notna().all().all()

assert not product_claims.duplicated(
    ["product_id", "claim_text", "claim_category"]
).any()

# Check that each excerpt exists in the original collected text.
assert all(
    row.claim_text.casefold() in row.original_key_benefits.casefold()
    for row in product_claims.itertuples()
)

output_path = (
    PROJECT_ROOT / "data/tableau/product_claims.csv"
)
product_claims.to_csv(output_path, index=False)

print("Products covered:", product_claims["product_id"].nunique())
print("Claim–category assignments:", len(product_claims))
print("\nPRODUCTS PER CLAIM CATEGORY")
print(
    product_claims.groupby("claim_category")["product_id"]
    .nunique()
    .sort_values(ascending=False)
)
print("\nSaved to:", output_path)